# 顧客分析｜顧客分級與輪廓洞察

商業問題：  
顧客等級與價值分析  
分析方法：  
- 依會員等級彙總消費總額、會員人數與平均客單價
- 比較不同會員等級的消費貢獻與客單價差異

In [5]:
SELECT Customer_Tier AS [客戶等級],
       ROUND(SUM(Total_Spent),2) AS [消費總額],
       COUNT(Customer_ID) AS [會員人數],
       ROUND(AVG(Total_Spent),2) AS [平均消費金額]
  FROM india_ecom.dbo.customers
 GROUP BY Customer_Tier
 ORDER BY [客戶等級];

(3 個資料列受到影響)

客戶等級     | 消費總額          | 會員人數  | 平均消費金額   
---------+---------------+-------+----------
Gold     | 232434570.64  | 6692  | 34733.2  
Platinum | 4455365021.57 | 27349 | 162907.79
Silver   | 53766763.35   | 5959  | 9022.78  
(3 個資料列)

總執行時間: 00:00:00.912

分析結果：  
Platinum 會員人數最多、消費總額與平均消費金額也遠高於 Gold 與 Silver ； Silver 人數最少，平均消費金額僅約九千元，明顯偏低。三個等級的消費表現差異明顯， Platinum 會員的平均消費與消費總額均較高，顯示高等級會員是主要的高價值客群。

商業問題：  
RFM 顧客價值分級  
分析方法：  
- 計算各會員的最近下單日、訂單數與總金額

In [ ]:
SELECT TOP(5000)Customer_ID, --因檔案過大，篩選前 5000 筆會員作為數據呈現，實際符合條件共 39726 人
       MAX(Order_Date) AS [最近下單日],
       COUNT(Order_ID) AS [訂單數],
       ROUND(SUM(Total_Amount),2) AS [總金額]
  FROM india_ecom.dbo.sales
 WHERE Order_Status='Delivered'
 GROUP BY Customer_ID
 ORDER BY Customer_ID;

(5000 個資料列受到影響)

Customer_ID  | 最近下單日      | 訂單數 | 總金額      
-------------+------------+-----+----------
CUST00000001 | 2026-01-12 | 5   | 40098.48 
CUST00000002 | 2026-02-13 | 8   | 63676.24 
CUST00000004 | 2025-09-28 | 5   | 288324.84
CUST00000005 | 2025-12-13 | 4   | 69265.75 
CUST00000006 | 2026-02-15 | 6   | 195157.42
CUST00000007 | 2026-01-09 | 7   | 216730.17
CUST00000008 | 2026-06-16 | 7   | 149633.69
CUST00000009 | 2026-01-27 | 7   | 339103.94
CUST00000010 | 2025-11-11 | 6   | 192737.1 
CUST00000011 | 2026-02-21 | 8   | 134681.04
CUST00000012 | 2026-06-02 | 4   | 415607.49
CUST00000013 | 2025-11-21 | 6   | 77308.68 
CUST00000014 | 2026-06-02 | 5   | 131378.22
CUST00000015 | 2026-06-30 | 4   | 81482.83 
CUST00000016 | 2026-05-31 | 8   | 86078.26 
CUST00000017 | 2026-06-11 | 5   | 208838.18
CUST00000018 | 2025-07-10 | 3   | 107100.8 
CUST00000019 | 2026-05-03 | 8   | 97966.36 
CUST00000020 | 2026-04-21 | 9   | 225738.85
CUST00000021 | 2024-12-21 | 2   | 27456.26 
CUST00000022 | 

分析結果：  
依 Customer_ID 彙總每位客戶的最近下單日（ Recency ）、訂單數( Frequency )與總金額( Monetary )，對應到 RFM 分析架構，可作為評估客戶價值的基礎資料，用來辨識高頻率、高消費的客戶，並作為後續客戶分群或忠誠度分析的依據。

商業問題：  
不同客群的商品類別消費差異  
分析方法：  
- 依年齡層、性別、商品類別分組計算個別營收
- 比較不同客群的商品類別營收差異

In [11]:
SELECT c.Age_Group,
       c.Gender,
       p.Category,
       ROUND(SUM(s.Total_Amount),2) AS [營收]
  FROM india_ecom.dbo.sales AS s
  JOIN india_ecom.dbo.products AS p
    ON s.Product_ID=p.Product_ID
  JOIN india_ecom.dbo.customers as c
    ON s.Customer_ID=c.Customer_ID
 GROUP BY c.Age_Group,c.Gender,p.Category
 ORDER BY c.Age_Group,c.Gender,p.Category;

(36 個資料列受到影響)

Age_Group | Gender | Category    | 營收          
----------+--------+-------------+-------------
18-25     | female | Books       | 9377699.01  
18-25     | female | Electronics | 662781866.04
18-25     | female | Fashion     | 68533641.04 
18-25     | male   | Books       | 9285335.95  
18-25     | male   | Electronics | 669066793.02
18-25     | male   | Fashion     | 68844762.93 
26-35     | female | Electronics | 953030337.6 
26-35     | female | Fashion     | 97984640.02 
26-35     | female | Sports      | 238564918.71
26-35     | male   | Electronics | 939654371.39
26-35     | male   | Fashion     | 98315749.5  
26-35     | male   | Sports      | 237274226.72
36-45     | female | Electronics | 546361900.57
36-45     | female | Grocery     | 10406395.89 
36-45     | female | Home        | 179514037.06
36-45     | male   | Electronics | 535454737.85
36-45     | male   | Grocery     | 10362194.01 
36-45     | male   | Home        | 177596251.31
46-55     | female | Beau

分析結果：  
資料依「年齡層、性別、商品類別」三欄分組呈現各組合的營收加總，共36組。可觀察到不同年齡層對應到不同的商品類別組合，同一年齡層內男女營收相近，商品類別選擇也一致（ 18-25 歲為書籍、電子、時尚， 26-35 歲為電子、時尚、運動， 36-45 歲為電子、雜貨、家居， 46以上歲普遍為美妝、雜貨、家居），各組合金額已按個別加總呈現。

商業問題：  
地區顧客分布與消費力分析  
分析方法：  
- 以`State`分組計算會員數與平均消費  
- 比較各地區的會員規模與消費表現

In [1]:
SELECT c.State,
       COUNT(distinct c.Customer_ID) AS [會員數],
       ISNULL(ROUND(AVG(s.Total_Amount),2),0) AS [平均訂單金額]
  FROM india_ecom.dbo.customers AS c
  LEFT JOIN india_ecom.dbo.sales AS s
    ON c.Customer_ID=s.Customer_ID
 GROUP BY c.State
 ORDER BY [會員數] DESC;

警告: 彙總或其他 SET 作業已刪除 Null 值。
(10 個資料列受到影響)

State       | 會員數  | 平均訂單金額  
------------+------+---------
UP          | 5218 | 23546.66
Haryana     | 5124 | 23553.37
Rajasthan   | 5057 | 23914.81
Punjab      | 4083 | 23745.81
Maharashtra | 4050 | 23778.58
Gujarat     | 4026 | 23804.29
Tamil Nadu  | 3170 | 23773.02
West Bengal | 3140 | 23461.18
Delhi       | 3116 | 24301.85
Karnataka   | 3016 | 23397.85
(10 個資料列)

總執行時間: 00:00:17.536

分析結果：  
依會員數排序各邦的會員規模後發現，UP、Haryana、Rajasthan 的會員數最多，皆超過 5,000 人；但各邦平均訂單金額差異不大，多落在 23,000～24,300 元之間，Delhi 雖會員規模較小，平均訂單金額則最高。整體而言，各邦的會員規模與平均訂單金額沒有呈現明顯一致的趨勢，會員數較多的地區不一定具有較高的平均訂單金額。